In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

from shapely.geometry import Point

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent                    # QSE-Tutorial/
DATA_ROOT  = REPO_ROOT / "Data"
U5_PATH = DATA_ROOT / "Shapefiles-2022" / "Berlin" / "TransportNetworkParts2006U5"
ARSW_DIR       = REPO_ROOT / "ARSW2015" / "ARSW2015-toolkit" / "shapefile"
TRANSPORT_DIR = U5_PATH

METRIC_CRS = "EPSG:25833"
WALKING_SPEED_M_MIN = 5000 / 60  # 5 km/h converted to meters per minute (83.33 m/min)

In [ ]:
# Set up for plotting
# Params for clean and minimalistic plots

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    #'figure.figsize': (6*2, 2*4.5),
    'font.size': 16.0,
    'font.family': 'serif',
    'font.serif': 'Palatino',
    'axes.titlesize': 'medium',
    'figure.titlesize': 'large',
    'legend.fontsize': 'medium',
    # dpi for high-res output
    'figure.dpi': 100,
    'savefig.dpi': 300,
    # Tight layout by default
    'figure.autolayout': True,
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{siunitx}[=v2]",
})

PLOT_ROOT = Path.cwd() / "Plots"
Path.mkdir(PLOT_ROOT, exist_ok=True)

In [ ]:
streets = gpd.read_file(U5_PATH / "Streets.shp").to_crs(epsg=25833).reset_index(drop=True)
# Extract unique street nodes (junctions)
start_points = streets.geometry.apply(lambda line: Point(line.coords[0]))
end_points = streets.geometry.apply(lambda line: Point(line.coords[-1]))
all_points = pd.concat([start_points, end_points])

# We reset index to guarantee unique sequence IDs for the street nodes
street_nodes = gpd.GeoDataFrame(geometry=all_points, crs=streets.crs).drop_duplicates(subset='geometry').reset_index(drop=True)
street_nodes['node_id'] = [f"street_node_{i}" for i in range(len(street_nodes))]

In [ ]:
allblocks_gdf = gpd.read_file(ARSW_DIR / "BerlinAllBlocks.shp").to_crs(epsg=25833).reset_index(drop=True)
green_gdf = gpd.read_file(ARSW_DIR / "BerlinGreen.shp").to_crs(epsg=25833).reset_index(drop=True)
water_gdf = gpd.read_file(ARSW_DIR / "BerlinWater.shp").to_crs(epsg=25833).reset_index(drop=True)

In [ ]:
blocks_gdf = gpd.read_file(ARSW_DIR / "Berlin4matlab.shp")
blocks_gdf = blocks_gdf[blocks_gdf.geometry.notnull() & ~blocks_gdf.geometry.is_empty].reset_index(drop=True)

def is_valid_coordinate_geom(geom):
    try:
        b = geom.bounds
        if len(b) != 4:
            return False
        return not (np.any(np.isnan(b)) or np.any(np.isinf(b)))
    except Exception:
        return False

# Drop corrupt coordinate geometries
blocks_gdf = blocks_gdf[blocks_gdf.geometry.apply(is_valid_coordinate_geom)].reset_index(drop=True)

# Repair geometry topologies safely in original CRS
blocks_gdf['geometry'] = blocks_gdf.geometry.make_valid()

# Project to metric CRS
blocks_gdf = blocks_gdf.to_crs(epsg=25833)

# Filter again to drop any nulls, empties, or corrupt geometries introduced by coordinate reprojection
blocks_gdf = blocks_gdf[blocks_gdf.geometry.notnull() & ~blocks_gdf.geometry.is_empty].reset_index(drop=True)
blocks_gdf = blocks_gdf[blocks_gdf.geometry.apply(is_valid_coordinate_geom)].reset_index(drop=True)

# Assign centroid_id directly to original block polygons so we can map results geographically later
blocks_gdf['centroid_id'] = [f"centroid_{i}" for i in range(len(blocks_gdf))]

centroids = blocks_gdf.geometry.centroid

centroids_gdf = gpd.GeoDataFrame(
    blocks_gdf.drop(columns=['geometry']),
    geometry=centroids,
    crs=blocks_gdf.crs
)

# Snap centroids to the nearest street node (Street-Walk Connector)
snapped_centroids = gpd.sjoin_nearest(
    centroids_gdf,
    street_nodes,
    how="left",
    distance_col="distance_to_node"
)
# In case of equidistant matches, keep only the first connection
snapped_centroids = snapped_centroids.groupby('centroid_id').first().reset_index()
snapped_centroids['travel_time_min'] = snapped_centroids['distance_to_node'] / WALKING_SPEED_M_MIN

In [ ]:
# Define filepaths for both Entrances and Stops/Platforms
entrances_files = {
    "Bus":   TRANSPORT_DIR / "BusEntrance.shp",
    "Tram":  TRANSPORT_DIR / "TramEntrance.shp",
    "SBahn": TRANSPORT_DIR / "SBahnEntrance.shp",
    "UBahn": TRANSPORT_DIR / "UBahnEntrance.shp",
}

stops_files = {
    "Bus":   TRANSPORT_DIR / "Bus2006_stops.shp",
    "Tram":  TRANSPORT_DIR / "Tram2006_stops.shp",
    "SBahn": TRANSPORT_DIR / "SBahn2006_stops.shp",
    "UBahn": TRANSPORT_DIR / "UBahn2006_stops.shp",
}

line_files = {
    "Bus":   TRANSPORT_DIR / "Bus2006_lines.shp",
    "Tram":  TRANSPORT_DIR / "Tram2006_lines.shp",
    "SBahn": TRANSPORT_DIR / "SBahn2006_lines.shp",
    "UBahn": TRANSPORT_DIR / "UBahn2006_lines.shp",
}

entrances_gdfs = {}
stops_gdfs = {}
lines_gdfs = {}

for mode in entrances_files.keys():
    entrances_gdfs[mode] = gpd.read_file(entrances_files[mode]).to_crs(epsg=25833).reset_index(drop=True)
    stops_gdfs[mode] = gpd.read_file(stops_files[mode]).to_crs(epsg=25833).reset_index(drop=True)
    lines_gdfs[mode] = gpd.read_file(line_files[mode]).to_crs(epsg=25833).reset_index(drop=True)

In [ ]:
# Load json with entrances
import json

with open(TRANSPORT_DIR / "osm_entrences.json", "r") as f:
    entrances_data = json.load(f)

entrences_df = pd.DataFrame(entrances_data['elements'])
entrences_gdf = gpd.GeoDataFrame(
    entrences_df,
    geometry=gpd.points_from_xy(entrences_df['lon'], entrences_df['lat']),
    crs="EPSG:4326"
).to_crs(epsg=25833).reset_index(drop=True)

In [ ]:
new_U5_stations = [
    {'Id': 171, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.4080828, 52.5188425)},  # Rotes Rathaus station
    {'Id': 172, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.3988844, 52.5172626)},  # Museum Island station
    {'Id': 173, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.3888200, 52.5169884)},  # Unter den Linden station
    {'Id': 174, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.3809897, 52.5166047)},  # Brandenburger Tor station
    {'Id': 175, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.3729545, 52.5201123)},  # Bundestag station
    {'Id': 176, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': Point(13.3701242, 52.5252819)},  # Berlin Hauptbahnhof station
]

# Create GeoDataFrame for new U5 stations
new_U5_stations_gdf = gpd.GeoDataFrame(new_U5_stations,
                                      geometry='geometry',
                                      crs="EPSG:4326").to_crs(epsg=25833).reset_index(drop=True)
# Append new U5 stations to existing stops GeoDataFrame
stops_gdfs['UBahn'] = pd.concat([stops_gdfs['UBahn'], new_U5_stations_gdf], ignore_index=True)


In [ ]:
# Expand new_U5_stations_gdf to circles with radius of 150 meters
new_U5_stations_gdf['geometry'] = new_U5_stations_gdf.geometry.buffer(180)

# Find all entrences_gdf that intersect with the new U5 station buffers
new_U5_entrances = gpd.sjoin(entrences_gdf, new_U5_stations_gdf, how="inner", predicate='intersects')

In [ ]:
len(new_U5_entrances) # Should be 30 - manually counted

In [ ]:
new_U5_entrances["Id"] = 0
# Drop all other columns except geometry and Id
new_U5_entrances = new_U5_entrances[["Id", "geometry"]].reset_index(drop=True)

In [ ]:
# Join new_U5_entrances to existing entrances_gdfs["UBahn"]
entrances_gdfs['UBahn'] = pd.concat([entrances_gdfs['UBahn'], new_U5_entrances], ignore_index=True).reset_index(drop=True)
entrances_gdfs['UBahn']

In [ ]:
# Plot the map of stops with type-specific colors and a basemap

import contextily as ctx
import seaborn as sns

plt.figure(figsize=(12, 10))
stops_gdfs["UBahn"].plot(
    ax=plt.gca(),
    markersize=20,
    alpha=0.7,
    color='grey',
)

stops_gdfs['UBahn'][stops_gdfs['UBahn']['NEAR_FID'] == 0].plot(
    ax=plt.gca(),
    markersize=20,
    alpha=0.7,
    color='firebrick'
)
ctx.add_basemap(plt.gca(), crs=stops_gdfs['UBahn'].crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.8, zorder=0)
plt.title("Public Transport Stops in Berlin (2006) + U5")
# Remove axis ticks and spines for a cleaner look
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(PLOT_ROOT / "berlin_ubahn_stops_2006U5.png", dpi=300, transparent=True)
plt.show()



In [ ]:
# Plot the map of stops with type-specific colors and a basemap

import contextily as ctx
import seaborn as sns

plt.figure(figsize=(12, 10))
new_U5_entrances.plot(
    ax=plt.gca(),
    markersize=25,
    alpha=0.8,
    color='royalblue',
)

stops_gdfs['UBahn'][stops_gdfs['UBahn']['NEAR_FID'] == 0].plot(
    ax=plt.gca(),
    markersize=20,
    alpha=0.8,
    color='firebrick'
)
ctx.add_basemap(plt.gca(), crs=stops_gdfs['UBahn'].crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6, zorder=0)
plt.title("Entrances of New U5 Stations in Berlin")
# Remove axis ticks and spines for a cleaner look
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(PLOT_ROOT / "berlin_U5_entrances.png", dpi=300, transparent=True)
plt.show()

In [ ]:
# Find the nearest stop to the following point
point = Point(13.4139082, 52.5215849)
point_gdf = gpd.GeoDataFrame([{'Id': 0, 'NEAR_FID': 0, 'NEAR_DIST': 0, 'NEAR_X': 0, 'NEAR_Y': 0, 'geometry': point}], geometry='geometry', crs="EPSG:4326").to_crs(epsg=25833).reset_index(drop=True)

nearest_id = stops_gdfs['UBahn'].geometry.distance(point_gdf.geometry.iloc[0]).idxmin()
nearest_stop = stops_gdfs['UBahn'].iloc[nearest_id]
nearest_stop


In [ ]:
# Plot the map of stops with type-specific colors and a basemap

import contextily as ctx
import seaborn as sns

plt.figure(figsize=(12, 10))
new_U5_entrances.plot(
    ax=plt.gca(),
    markersize=25,
    alpha=0.8,
    color='royalblue',
)

stops_gdfs['UBahn'][stops_gdfs['UBahn']['NEAR_FID'] == 0].plot(
    ax=plt.gca(),
    markersize=20,
    alpha=0.8,
    color='firebrick'
)

nearest_stop_gdf = gpd.GeoDataFrame([nearest_stop], geometry='geometry', crs=stops_gdfs['UBahn'].crs)
nearest_stop_gdf.plot(
    ax=plt.gca(),
    markersize=100,
    alpha=0.9,
    color='gold',
    edgecolor='black')

ctx.add_basemap(plt.gca(), crs=stops_gdfs['UBahn'].crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6, zorder=0)
plt.title("Entrances of New U5 Stations in Berlin")
# Remove axis ticks and spines for a cleaner look
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.tight_layout()
#plt.savefig(PLOT_ROOT / "berlin_U5_entrances.png", dpi=300, transparent=True)
plt.show()

In [ ]:
# Alex has been found
# Find a SMOOTH line that goes through all the stops with NEAR_FID == 0 (the new U5 stations)

new_U5_stops = stops_gdfs['UBahn'][stops_gdfs['UBahn']['NEAR_FID'] == 0]
# Append nearest_stop (Id= 180 to new_U5_stops)
new_U5_stops = pd.concat([new_U5_stops, nearest_stop_gdf], ignore_index=True).reset_index(drop=True)


In [ ]:
from shapely.geometry import LineString
from scipy.interpolate import splprep, splev

# Build a smooth approximation of the U5 alignment from the station stop points.
# The stops are ordered by Id, which matches the intended west-east sequence.
ordered_u5_stops = new_U5_stops.sort_values("Id").reset_index(drop=True)
coords = np.array([(geom.x, geom.y) for geom in ordered_u5_stops.geometry])

if len(coords) < 2:
    raise ValueError("Need at least two points to build a smooth U5 line.")

# Fit a parametric cubic spline through the stop coordinates.
k = min(3, len(coords) - 1)
tck, _ = splprep(coords.T, s=0, k=k)

# Sample the spline densely to obtain a smooth line geometry.
u_fine = np.linspace(0, 1, 300)
smooth_x, smooth_y = splev(u_fine, tck)
new_U5_line_geom = LineString(zip(smooth_x, smooth_y))

new_U5_line = gpd.GeoDataFrame(
    {"name": ["U5_approximated_trajectory"]},
    geometry=[new_U5_line_geom],
    crs=ordered_u5_stops.crs,
)

new_U5_line

In [ ]:
# Plot the map of stops with type-specific colors and a basemap

import contextily as ctx
import seaborn as sns

plt.figure(figsize=(12, 10))
new_U5_line.plot(
    ax=plt.gca(),
    linewidth=7,
    alpha=0.7,
    color='#7e5330',
)

new_U5_stops.plot(
    ax=plt.gca(),
    markersize=200,
    alpha=1,
    #color='#7e5330',
    edgecolor='#7e5330',
    facecolor='#ffffff',
    linewidth=4,
    zorder=2
)

ctx.add_basemap(plt.gca(), crs=stops_gdfs['UBahn'].crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6, zorder=0)
plt.title("Entrances of New U5 Stations in Berlin")
# Remove axis ticks and spines for a cleaner look
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(PLOT_ROOT / "berlin_U5.png", dpi=300, transparent=True)
plt.show()

In [ ]:
from shapely.ops import substring

# Split the approximated U5 trajectory into stop-to-stop segments.
# With 7 stops this creates 6 ordered segments.
line_geom = new_U5_line.geometry.iloc[0]
ordered_u5_stops = new_U5_stops.sort_values("Id").reset_index(drop=True)

# Distance of each stop along the smoothed trajectory.
stop_distances = ordered_u5_stops.geometry.apply(line_geom.project).to_numpy()

segments = []
for i in range(len(ordered_u5_stops) - 1):
    d0 = float(stop_distances[i])
    d1 = float(stop_distances[i + 1])
    start_d, end_d = sorted([d0, d1])

    seg_geom = substring(line_geom, start_d, end_d)
    segments.append(
        {
            "segment_id": f"U5_segment_{i+1}",
            "from_stop_id": int(ordered_u5_stops.iloc[i]["Id"]),
            "to_stop_id": int(ordered_u5_stops.iloc[i + 1]["Id"]),
            "length_m": seg_geom.length,
            "geometry": seg_geom,
        }
    )

new_U5_segments = gpd.GeoDataFrame(segments, geometry="geometry", crs=new_U5_line.crs)
new_U5_segments

In [ ]:
# Drop segment_id, from_stop_id, to_stop_id
new_U5_segments = new_U5_segments.drop(columns=['segment_id', 'from_stop_id', 'to_stop_id'], errors='ignore')
# rename length_m to Length
new_U5_segments = new_U5_segments.rename(columns={'length_m': 'Length'})
new_U5_segments['Id'] = 0
new_U5_segments['Minutes'] = 0.0

# Find in the original lines_gdfs['UBahn'] segments that have similar length and match the Minutes
for idx, row in new_U5_segments.iterrows():
    length = row['Length']
    diff = np.abs(lines_gdfs['UBahn']['Length'] - length)
    closest_idx = diff.idxmin()
    closest_segment = lines_gdfs['UBahn'].iloc[closest_idx]
    new_U5_segments.at[idx, 'Minutes'] = closest_segment['Minutes']
    

In [ ]:
# Join the new U5 segments to the existing lines_gdfs['UBahn']
lines_gdfs['UBahn'] = pd.concat([lines_gdfs['UBahn'], new_U5_segments], ignore_index=True).reset_index(drop=True)
lines_gdfs['UBahn']

In [ ]:
from pathlib import Path


def overwrite_shapefile_family(gdf, target_shp_path):
    stem = target_shp_path.with_suffix("")
    for existing_file in stem.parent.glob(stem.name + ".*"):
        existing_file.unlink()
    gdf.to_file(target_shp_path, driver="ESRI Shapefile", index=False, encoding="UTF-8")
    return sorted(path.name for path in stem.parent.glob(stem.name + ".*"))


ubahn_entrance_path = TRANSPORT_DIR / "UBahnEntrance.shp"
ubahn_stops_path = TRANSPORT_DIR / "UBahn2006_stops.shp"
ubahn_lines_path = TRANSPORT_DIR / "UBahn2006_lines.shp"

updated_entrances = entrances_gdfs["UBahn"][["Id", "geometry"]].copy()
updated_stops = stops_gdfs["UBahn"][["Id", "geometry"]].copy()
updated_lines = lines_gdfs["UBahn"][["Id", "Length", "Minutes", "geometry"]].copy()
updated_lines["Id"] = updated_lines["Id"].astype("int64")
updated_lines["Length"] = updated_lines["Length"].astype(float)
updated_lines["Minutes"] = updated_lines["Minutes"].astype(float)

written_entrance_files = overwrite_shapefile_family(updated_entrances, ubahn_entrance_path)
written_stop_files = overwrite_shapefile_family(updated_stops, ubahn_stops_path)
written_line_files = overwrite_shapefile_family(updated_lines, ubahn_lines_path)

reloaded_entrances = gpd.read_file(ubahn_entrance_path)
reloaded_stops = gpd.read_file(ubahn_stops_path)
reloaded_lines = gpd.read_file(ubahn_lines_path)

assert list(reloaded_entrances.columns) == ["Id", "geometry"]
assert list(reloaded_stops.columns) == ["Id", "geometry"]
assert list(reloaded_lines.columns) == ["Id", "Length", "Minutes", "geometry"]

assert len(reloaded_entrances) == len(updated_entrances)
assert len(reloaded_stops) == len(updated_stops)
assert len(reloaded_lines) == len(updated_lines)

print("Wrote entrance files:", written_entrance_files)
print("Wrote stop files:", written_stop_files)
print("Wrote line files:", written_line_files)
print("Counts:", len(reloaded_entrances), len(reloaded_stops), len(reloaded_lines))
print("CRS:", reloaded_entrances.crs, reloaded_stops.crs, reloaded_lines.crs)

In [ ]:

blocks_gdf.plot(figsize=(10,10), color='lightgrey', edgecolor='none')
green_gdf.plot(ax=plt.gca(), color='forestgreen', edgecolor='none', alpha=0.5)
water_gdf.plot(ax=plt.gca(), color='royalblue', edgecolor='none', alpha=0.5)
streets.plot(ax=plt.gca(), color='beige', linewidth=0.5)